# RouteRanker – Module 1: Data Collection and Dataset Understanding
**Goal:** collect the GTFS files, understand their structure and relationships, and document data quality. 
**Not in scope:** cleaning, feature engineering or any model. Nothing here deletes or changes data.

**Honest status:** the Lepton Software / Datarade dataset named in the project document is a *paid* listing and could not be downloaded
(see `reports/module1_data_sources.md`). Open GTFS feeds for **7 cities** were collected instead. GTFS has **no passenger counts** – none are created here.

In [12]:
import os, sys, glob
import pandas as pd
# run from the project root regardless of where Jupyter was started
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
from src import inspect_gtfs as ig
pd.set_option("display.max_columns", 30, "display.width", 200)

CITY = "pune"          # <- change to any folder name in data/raw/ (ahmedabad, bengaluru, chennai, delhi, hyderabad, kochi, pune)
CITY_DIR = f"data/raw/{CITY}"
print("project root:", os.getcwd())

project root: c:\Users\ksaty\Desktop\RouteRanker


## Task 1 – Collected data and folder organisation

In [13]:
rows = []
for d in sorted(glob.glob("data/raw/*")):
    if os.path.isdir(d) and os.path.basename(d) not in ("sample_city", "demo"):
        z = glob.glob(f"{d}/*.zip")
        rows.append(dict(city=os.path.basename(d), zip=os.path.basename(z[0]) if z else "-",
                         zip_MB=round(os.path.getsize(z[0]) / 1e6, 1) if z else None,
                         txt_files=len(glob.glob(f"{d}/*.txt"))))
pd.DataFrame(rows)

,city,zip,zip_MB,txt_files
0,ahmedabad,ahmedabad_gtfs.zip,4.6,12
1,bengaluru,bengaluru_gtfs.zip,44.4,12
2,chennai,chennai_gtfs.zip,9.8,13
3,delhi,delhi_gtfs.zip,19.8,6
4,hyderabad,hyderabad_gtfs.zip,10.5,7
5,kochi,kochi_gtfs.zip,5.5,9
6,pune,pune_gtfs.zip,4.9,8


The original ZIPs stay untouched in `data/raw/<city>/`; `ensure_extracted` only extracts the `.txt` files next to them.

## Task 3 – Load the files with pandas

In [14]:
print(ig.ensure_extracted(CITY_DIR))
tables = ig.load_city(CITY_DIR)           # pd.read_csv on every available GTFS .txt (ID columns read as text)
pd.DataFrame([dict(file=f"{n}.txt", kind="required" if n in ig.REQUIRED else "optional", rows=len(df), columns=df.shape[1])
              for n, df in tables.items()])

already extracted


,file,kind,rows,columns
0,routes.txt,required,315,5
1,trips.txt,required,10128,6
2,stop_times.txt,required,429302,6
3,stops.txt,required,6745,4
4,calendar.txt,required,1,10
5,shapes.txt,optional,256823,4
6,agency.txt,optional,1,6
7,feed_info.txt,optional,1,6


## Task 2 – Structure: columns, data types and sample values

In [15]:
for name in ["routes", "trips", "stop_times", "stops", "calendar"]:
    df = tables[name]
    print(f"\n===== {name}.txt  ->  {len(df):,} rows x {df.shape[1]} columns")
    display(pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum(), "example": df.iloc[0]}))
    display(df.head(3))


===== routes.txt  ->  315 rows x 5 columns


,dtype,non_null,example
route_id,str,315,100
agency_id,str,315,PMPML
route_short_name,str,315,100
route_long_name,str,315,Hinjawadi Maan Phase 3 ⇆ Ma Na Pa Dengle Pul N...
route_type,int64,315,3


,route_id,agency_id,route_short_name,route_long_name,route_type
0,100,PMPML,100,Hinjawadi Maan Phase 3 ⇆ Ma Na Pa Dengle Pul N...,3
1,103,PMPML,103,Kothrud Depot ⇆ Katraj,3
2,104,PMPML,104,Dsk Vishwa ⇆ Shivaji Nagar,3



===== trips.txt  ->  10,128 rows x 6 columns


,dtype,non_null,example
route_id,str,10128,108
service_id,str,10128,WEEKDAY
trip_id,str,10128,trip_18_0
trip_headsign,str,10128,Sutardara
direction_id,int64,10128,0
shape_id,str,10128,18


...

,dtype,non_null,example
stop_id,str,6745,106793
stop_name,str,6745,Pune Station Moledina Road
stop_lat,float64,6745,18.527575
stop_lon,float64,6745,73.873245


,stop_id,stop_name,stop_lat,stop_lon
0,106793,Pune Station Moledina Road,18.527575,73.873245
1,319,Income Tax Office,18.525400,73.876701
2,320,Gpo,18.523537,73.875557



===== calendar.txt  ->  1 rows x 10 columns


,dtype,non_null,example
service_id,str,1,WEEKDAY
monday,int64,1,1
tuesday,int64,1,1
wednesday,int64,1,1
thursday,int64,1,1
friday,int64,1,1
saturday,int64,1,1
sunday,int64,1,1
start_date,int64,1,20261007
end_date,int64,1,20270405


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,WEEKDAY,1,1,1,1,1,1,1,20261007,20270405


### Optional files (frequencies.txt, shapes.txt, ...)

In [16]:
for n in ig.OPTIONAL:
    if n in tables:
        print(f"{n}.txt available: {len(tables[n]):,} rows -> {ig.USEFULNESS[n]}")
    else:
        print(f"{n}.txt NOT available in {CITY}")

calendar_dates.txt NOT available in pune
frequencies.txt NOT available in pune
shapes.txt available: 256,823 rows -> Exact road geometry of a route; gives true route length/maps (stop-to-stop distance is only an approximation).
agency.txt available: 1 rows -> Operator name/contact; useful for documentation and multi-operator cities.
feed_info.txt available: 1 rows -> Feed publisher, version and validity dates; useful to judge how current the data is.


## Task 4 – Data-quality inspection (report only – nothing is removed or filled)

In [17]:
q = pd.DataFrame(ig.quality_checks(CITY, tables))
print("checks run:", len(q), "| checks with at least one finding:", int((q["count"] > 0).sum()))
q[q["count"] > 0][["file", "check", "count", "pct", "note"]]

checks run: 30 | checks with at least one finding: 2


,file,check,count,pct,note
21,stop_times.txt,time >= 24:00:00 (valid GTFS: trip runs past m...,179,0.04,Valid in GTFS; must be handled when extracting...
29,calendar.txt,service runs all 7 days (no weekday/weekend di...,1,100.00,1 service_id(s) in calendar.txt


In [18]:
# missing values per column in the required files
for name in ig.REQUIRED:
    miss = tables[name].isna().sum()
    print(name, "->", dict(miss[miss > 0]) or "no missing values")

routes -> no missing values
trips -> no missing values
stop_times -> no missing values
stops -> no missing values
calendar -> no missing values


## Task 5 – Relationships between the files
`routes ─route_id→ trips ─trip_id→ stop_times ─stop_id→ stops`, and `trips ─service_id→ calendar`.

In [19]:
trip = tables["trips"].iloc[0]
print("route:", trip.route_id, "| trip:", trip.trip_id, "| service:", trip.service_id)
one = (tables["stop_times"][tables["stop_times"].trip_id == trip.trip_id]
       .merge(tables["stops"][["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id", how="left")
       .sort_values("stop_sequence"))
one[["stop_sequence", "arrival_time", "departure_time", "stop_id", "stop_name", "stop_lat", "stop_lon"]].head(10)

route: 108 | trip: trip_18_0 | service: WEEKDAY


,stop_sequence,arrival_time,departure_time,stop_id,stop_name,stop_lat,stop_lon
0,1,08:30:00,08:30:00,106793,Pune Station Moledina Road,18.527575,73.873245
1,2,08:31:00,08:31:00,319,Income Tax Office,18.525400,73.876701
2,3,08:33:09,08:33:09,320,Gpo,18.523537,73.875557
3,4,08:34:09,08:34:09,321,Collector Kacheri,18.524081,73.872966
4,5,08:35:09,08:35:09,322,Zilla Parishad,18.523402,73.869253
5,6,08:36:27,08:36:27,323,15 August Chowk,18.522294,73.866395
6,7,08:37:29,08:37:29,324,Apollo Talkies,18.519501,73.864601
7,8,08:38:45,08:38:45,325,Phadke Haud,18.518999,73.860298
8,9,08:40:05,08:40:05,326,Jijamatabaug,18.518704,73.857087
9,10,08:41:12,08:41:12,327,Vasant Talkies,18.518310,73.856217


In [20]:
pd.DataFrame([ig.relationship_stats(CITY, tables)]).T.rename(columns={0: CITY})

,pune
city,pune
routes,315
trips,10128
trips_per_route_median,22.0
trips_per_route_max,175
stop_times,429302
stops_per_trip_median,42.0
stops,6745
stop_times_matching_stops_pct,100.0
services_in_calendar,1


## Task 6 – Dataset inventory (all cities)
Generated by `python run_module1.py`; written to `reports/module1_file_inventory.csv` and `reports/module1_dataset_inventory.md`.

In [21]:
inv = pd.read_csv("reports/module1_file_inventory.csv")
req = inv[inv.kind == "required"].pivot(index="city", columns="file", values="records")
req

file,calendar.txt,routes.txt,stop_times.txt,stops.txt,trips.txt
city,,,,,
ahmedabad,1,978,646854,3235,14323
bengaluru,1,4434,1540686,9960,57836
chennai,12,5402,1389081,6542,48435
delhi,1,2964,3276092,6342,78515
hyderabad,1,1031,1132807,5028,44892
kochi,1,473,46648,3258,933
pune,1,315,429302,6745,10128


In [22]:
qual = pd.read_csv("reports/module1_quality_issues.csv")
qual[qual["count"] > 0].groupby("city").size().rename("checks with findings").to_frame()

,checks with findings
city,
ahmedabad,5
bengaluru,4
chennai,9
delhi,6
hyderabad,2
kochi,3
pune,2


## Limitations (carry forward to later modules)
* The Lepton/Datarade 23-city dataset was **not** obtained; 7 open feeds were used. Mumbai, Rajkot and Hubli-Dharwad have no open feed.
* GTFS has **no passenger counts, occupancy or live traffic**. Any later demand figure is an *estimate* and must be labelled as such.
* Most feeds cannot separate weekday from weekend; several schedules are expired (see the inventory report).
* A route with few scheduled trips is **not** automatically inefficient – operational context is needed.